# 📈 Caderno 03: Calibração Multivariada Quantitativa (PLSR e Random Forest)
**Projeto:** Detecção de Adulteração em Fertilizantes NPK via LIBS  
**Instituição:** Embrapa Instrumentação  

---

### Objetivos deste Caderno:
1. **Modelagem Quantitativa Contínua** do teor de adulteração (%) utilizando o espectro completo.
2. **Partial Least Squares Regression (PLSR):**
   - Otimização do número de Variáveis Latentes (LVs) via curvas de RMSEC e RMSECV (Leave-One-Group-Out CV).
   - Treinamento do modelo final de calibração.
3. **Quantificação da Incerteza de Medição Tipo A ($u_A = s / \sqrt{n}$)** para cada teor de adulteração.
4. **Modelagem Não Linear com Random Forest Regressor:**
   - Validação cruzada agrupada por pastilha.
   - Extração da importância dos comprimentos de onda para predição quantitativa.

In [ ]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.models import (
    otimizar_componentes_plsr,
    avaliar_plsr_final,
    avaliar_rf_regressor
)
from src.visualization import (
    plotar_curva_rmsec_rmsecv,
    plotar_calibracao_real_vs_predito
)

print("Módulos carregados com sucesso!")

## 1. Carregamento dos Dados Tratados

In [ ]:
pasta_proc = BASE_DIR / "data" / "processed"
caminho_vis = pasta_proc / "df_limpo_vis.parquet"

df_vis = pd.read_parquet(caminho_vis) if caminho_vis.exists() else None

if df_vis is not None:
    # Filtragem de amostras e seleção de features
    df_pls = df_vis[df_vis['Amostra_ID'] != 'i_'].copy()
    metadados = ['Amostra_ID', 'Num_Disparo', 'Nome_Arquivo', 'Canal_Optico', 'Teor_Adulteracao', 'Adulterado']
    
    cols_features = [c for c in df_pls.columns if c not in metadados and 273.0 <= float(c) <= 780.0]
    X = df_pls[cols_features].values.astype(float)
    y = df_pls['Teor_Adulteracao'].values.astype(float)
    grupos = df_pls['Amostra_ID'].values

    print(f"Matriz X: {X.shape} (N amostras x P comprimentos de onda)")
    print(f"Classes de teores presentes: {np.unique(y)}")
else:
    print("Aviso: execute primeiro o Caderno 01 para gerar os dados tratados.")

## 2. Otimização de Variáveis Latentes na PLS-Regression
Calculamos RMSEC (erro de calibração) e RMSECV (erro de validação cruzada com Leave-One-Group-Out) de 1 a 15 LVs.

In [ ]:
if df_vis is not None:
    df_lvs = otimizar_componentes_plsr(X, y, grupos, max_components=12)
    print("Tabela de Desempenho por Número de LVs:")
    display(df_lvs) if 'display' in dir() else print(df_lvs.to_string())

    # Gráfico de cotovelo RMSEC vs RMSECV
    fig_lvs = plotar_curva_rmsec_rmsecv(df_lvs)
    plt.show()

    # Seleciona número ótimo pelo mínimo RMSECV
    lv_otimo = int(df_lvs.loc[df_lvs['RMSECV'].idxmin()]['LVs'])
    print(f"Número ótimo de Variáveis Latentes sugerido: {lv_otimo}")

## 3. Modelo Final de PLSR e Incerteza de Medição Tipo A
Avaliamos a calibração com o número ótimo de LVs e calculamos a dispersão e incerteza $u_A = rac{s}{\sqrt{n}}$ por nível.

In [ ]:
if df_vis is not None:
    res_pls = avaliar_plsr_final(X, y, grupos, n_components=lv_otimo)

    print(f"R² de Validação Cruzada (LOGO-CV): {res_pls['R2_CV']:.4f}")
    print(f"RMSECV: {res_pls['RMSECV']:.2f}% de teor de adulteração")

    # Gráfico Real vs Predito
    fig_cal = plotar_calibracao_real_vs_predito(
        res_pls['y_real'], res_pls['y_pred_cv'],
        r2=res_pls['R2_CV'], rmse=res_pls['RMSECV'],
        titulo="Calibração PLSR (Leave-One-Group-Out CV)"
    )
    plt.show()

    print("
Tabela de Incertezas do Tipo A:")
    display(res_pls['tabela_incertezas']) if 'display' in dir() else print(res_pls['tabela_incertezas'].to_string())

## 4. Modelagem Não Linear com Random Forest Regressor
Comparação de desempenho utilizando Random Forest com validação cruzada por pastilha.

In [ ]:
if df_vis is not None:
    res_rf = avaliar_rf_regressor(
        X, y, grupos=grupos,
        n_estimators=200,
        max_features=0.33,
        random_state=42
    )

    print(f"Random Forest Regressor:")
    print(f"  R² (LOGO-CV): {res_rf['R2']:.4f}")
    print(f"  RMSE (LOGO-CV): {res_rf['RMSE']:.2f}%")

    fig_rf = plotar_calibracao_real_vs_predito(
        res_rf['y_real'], res_rf['y_pred'],
        r2=res_rf['R2'], rmse=res_rf['RMSE'],
        titulo="Calibração Random Forest (LOGO-CV)"
    )
    plt.show()

## 5. Comparativo Final de Modelos de Regressão

In [ ]:
if df_vis is not None:
    comparativo = pd.DataFrame([
        {"Modelo": f"PLS-Regression ({lv_otimo} LVs)", "R²_CV": round(res_pls['R2_CV'], 4), "RMSECV (%)": round(res_pls['RMSECV'], 2)},
        {"Modelo": "Random Forest (200 árvores)", "R²_CV": round(res_rf['R2'], 4), "RMSECV (%)": round(res_rf['RMSE'], 2)}
    ])
    display(comparativo) if 'display' in dir() else print(comparativo.to_string())